In [ ]:
import cv2
import numpy as np
import os
import skimage as ski
from tqdm import tqdm

In [ ]:
def ski_gauss_poisson_noise(
    img,
    exposure,
    gamma,
    blur,
    gaussian,
):
    # Randomize Values
    exposure = np.random.uniform(*exposure)
    gamma = np.random.uniform(*gamma)
    blur = int(np.random.uniform(*blur))
    gaussian = np.random.uniform(*gaussian)

    if blur % 2 == 0:
        blur += 1

    # OpenCV BGR → RGB
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

    # Normalize to [0,1]
    img = img.astype(np.float32) / 255.0

    # sRGB → linear
    img_lin = np.power(img, 2.2)

    # Gaussian Blur 
    img_lin = cv2.GaussianBlur(img_lin, (blur,blur), 0)

    # Exposure 
    img_lin *= exposure

    # Gamma
    img_lin = ski.exposure.adjust_gamma(img_lin, gamma=gamma)

    # Poisson noise
    img_lin = ski.util.random_noise(img_lin, mode="poisson")

    # Gaussian noise
    img_lin = ski.util.random_noise(img_lin, mode="gaussian", var=gaussian)

    # Prevent Full Clipping
    img_lin = np.maximum(img_lin, 0.0)
    img_lin = img_lin / (1.0 + img_lin)

    # Clip
    img_lin = np.clip(img_lin, 0.0, 1.0)

    # linear → sRGB
    img_out = np.power(img_lin, 1.0 / 2.2)

    # Back to uint8
    img_out = (img_out * 255).astype(np.uint8)

    # OpenCVRGB → OpenCV BGR
    img_out = cv2.cvtColor(img_out, cv2.COLOR_RGB2BGR)

    return img_out


In [ ]:
def motion_blur(img_bgr, k_range, angle_range=(0, 180)):

    k = int(np.random.randint(k_range[0], k_range[1] + 1))
    if k % 2 == 0:
        k += 1

    angle = float(np.random.uniform(angle_range[0], angle_range[1]))

    # Build a horizontal line kernel
    kernel = np.zeros((k, k), dtype=np.float32)
    kernel[k // 2, :] = 1.0

    # Rotate kernel by angle degrees
    center = (k / 2 - 0.5, k / 2 - 0.5)
    M = cv2.getRotationMatrix2D(center, angle, 1.0)
    kernel = cv2.warpAffine(kernel, M, (k, k), flags=cv2.INTER_LINEAR)
    
    # Normalize kernel so brightness doesn't change
    s = kernel.sum()
    if s > 0:
        kernel /= s

    # Convolve
    blurred = cv2.filter2D(img_bgr, ddepth=-1, kernel=kernel, borderType=cv2.BORDER_REFLECT101)
    return blurred

In [ ]:
input_dir = "dataset/test/images_original"
output_dir = "dataset/test/images"

os.makedirs(output_dir, exist_ok=True)

brightness_factor = 0.2
noise_std = 10

image_files = [
    f for f in os.listdir(input_dir)
    if f.lower().endswith((".jpg", ".png", ".jpeg"))
]

for filename in tqdm(image_files, desc="Darkening images", unit="img"):
    img_path = os.path.join(input_dir, filename)
    img = cv2.imread(img_path)

    if img is None:
        continue

    out_path = os.path.join(output_dir, filename)

    image = motion_blur(img, (0, 12))
    image = ski_gauss_poisson_noise(image, 
    exposure=(0.05, 0.1),
    gamma=(1.9, 2.1),
    blur = (5, 10),
    gaussian= (0.000000001, 0.0000000015))

    cv2.imwrite(out_path, image)

print("Done!")


Darkening images:  25%|██▌       | 193/771 [01:27<02:53,  3.33img/s]

In [ ]:
from tqdm import tqdm
import os
import cv2

input_dir = "dataset/valid/images_original"
output_dir = "dataset/valid/images"

os.makedirs(output_dir, exist_ok=True)

brightness_factor = 0.2
noise_std = 10

image_files = [
    f for f in os.listdir(input_dir)
    if f.lower().endswith((".jpg", ".png", ".jpeg"))
]

for filename in tqdm(image_files, desc="Darkening images", unit="img"):
    img_path = os.path.join(input_dir, filename)
    img = cv2.imread(img_path)

    if img is None:
        continue

    out_path = os.path.join(output_dir, filename)

    # Add Motion Blur First
    image = motion_blur(img, (0, 12))

    # Add Everything Else Next
    image = ski_gauss_poisson_noise(image, 
    exposure=(0.05, 0.1),
    gamma=(1.9, 2.1),
    blur = (5, 10),
    gaussian= (0.000000001, 0.0000000015)
    )

    cv2.imwrite(out_path, image)

print("Done!")


In [ ]:
from tqdm import tqdm
import os
import cv2

input_dir = "dataset/train/images_original"
output_dir = "dataset/train/images"

os.makedirs(output_dir, exist_ok=True)

brightness_factor = 0.2
noise_std = 10

image_files = [
    f for f in os.listdir(input_dir)
    if f.lower().endswith((".jpg", ".png", ".jpeg"))
]

for filename in tqdm(image_files, desc="Darkening images", unit="img"):
    img_path = os.path.join(input_dir, filename)
    img = cv2.imread(img_path)

    if img is None:
        continue

    out_path = os.path.join(output_dir, filename)

    image = motion_blur(img, (0, 12))
    image = ski_gauss_poisson_noise(image, 
    exposure=(0.05, 0.1),
    gamma=(1.9, 2.1),
    blur = (5, 10),
    gaussian= (0.000000001, 0.0000000015))

    cv2.imwrite(out_path, image)

print("Done!")
